# MethaneSET · multispectral

Two sensors share this tutorial: Sentinel-2 (13 bands, 10 m) and Landsat 8/9 (9 bands, 30 m on a 10 m grid). Change `SENSOR` below and the rest runs unchanged. The scenes are the plume chips of the finetune subset, with their three extra plume-free references.

Every figure carries its units and a scale bar: the pixel is 10 m for Sentinel-2 and 30 m for Landsat.

In [ ]:
SENSOR = "s2"   # "s2" or "l89"

In [ ]:
# Keep inline figures small; the text and code matter more than pixel count.

In [ ]:
HF_TOKEN = ""  #@param {type:"string"}

# Anonymous Hugging Face reads share the rate limit of the whole Colab fleet. With a free
# account token pasted above the limit is yours, and the notebook stops hitting 429.
# Create one at https://huggingface.co/settings/tokens (read access is enough).
if HF_TOKEN:
    import os
    os.environ["HF_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(HF_TOKEN, add_to_git_credential=False)
        print("Using your Hugging Face token.")
    except Exception as exc:
        print("Token not accepted:", exc)

In [ ]:
# Setup
import os, urllib.request
if not os.path.exists("common.py"):          # on Colab the notebook travels alone
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/IPL-UV/taco-methaneset/main/methaneset-codes/notebooks/common.py",
        "common.py")
try:
    import tacoreader, rasterio, sklearn, duckdb
except ImportError:
    %pip install "tacoreader>=2.4,<3" rasterio scikit-learn matplotlib pandas pyarrow duckdb
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import common as C
C.style()
C.versions()
print("dataset version:", C.dataset_version("methaneset-{SENSOR}-finetune"))

## The catalog

`level0` is the whole catalog as a table: one row per sample, no raster opened. The namespaces are `target:`, `detection:`, `site:` and `split`. On a local copy the query runs in DuckDB; on Hugging Face it falls back to pandas.

In [ ]:
pre = C.load(f"methaneset-{SENSOR}-pretraining")
ft = C.load(f"methaneset-{SENSOR}-finetune")
pre_df = pre.data.to_pandas()
ft_df = ft.data.to_pandas()
print("pretraining:", pre_df.shape[0], "samples")
print("finetune   :", ft_df.shape[0], "samples")

# A real SQL query on level0. On a local catalog DuckDB reads the Parquet directly.
# On Hugging Face we fall back to a pandas query so the cell still runs.
root = C.resolve(f"methaneset-{SENSOR}-finetune")
if root.startswith("http"):
    q = ft_df.query("`target:cloud` < 0.02 and `detection:ch4_fluxrate` > 1000")[
        ["id", "target:cloud", "detection:ch4_fluxrate"]].head(5)
else:
    import duckdb
    con = duckdb.connect()
    path = f"{root}/.tacocat/level0.parquet"
    q = con.execute(
        'SELECT id, "target:cloud", "detection:ch4_fluxrate" FROM read_parquet(?) '
        'WHERE "target:cloud" < 0.02 AND "detection:ch4_fluxrate" > 1000 LIMIT 5',
        [path],
    ).fetchdf()
print(q)

In [ ]:
print(sorted([c for c in ft_df.columns if c.startswith(("detection:", "target:"))]))
ft_df[["target:tile", "target:sensor", "target:sza", "target:vza", "target:saa", "target:raa",
         "detection:ch4_fluxrate", "site:country", "split"]].head(3)

## One scene from the finetune subset

The brightest plume below 5 % cloud. The target chip, its reference, the MARS enhancement stored with the sample (`ch4`, in ppb) and the plume mask are all read from the same sample id.

In [ ]:
f = ft_df
idx = f.sort_values("detection:ch4_fluxrate", ascending=False).query("`target:cloud` < 0.05").index[0]
s = ft.read(int(idx)); row = f.loc[idx]
print("flux [kg/h]:", row["detection:ch4_fluxrate"], "| country:", row["site:country"], "| SZA [deg]:", row["target:sza"])
tgt = C.read(s, "target"); bg0 = C.read(s, "bg0"); ch4 = C.read(s, "ch4", bands=1)
plume = C.read(s, "plume", bands=1)
print("target:", tgt.shape, tgt.dtype, "| ch4:", ch4.shape, ch4.dtype, "| plume:", np.unique(plume))

C.layers(s)   # dtype, nodata and shape of every layer in this sample

In [ ]:
fig, axes = C.panels(6, rows=2, kind="image")
dem = C.read(s, "dem", bands=1)

# (a) target RGB
C.imshow_layer(axes[0], C.to_rgb(tgt, C.SENSORS[SENSOR]["rgb"]), kind="enh",
               title="Target RGB", cbar=False, px_m=C.PIXEL_M[SENSOR])
swir = C.SENSORS[SENSOR]["swir"]
C.panel_label(axes[0], "a")

# (b) target B11 (1610 nm, 0-based index 11)
C.imshow_layer(axes[1], tgt[swir[0]], kind="enh", title="Target B11 [1610 nm]",
               label="reflectance", px_m=C.PIXEL_M[SENSOR])
C.panel_label(axes[1], "b")

# (c) bg0 B11
C.imshow_layer(axes[2], bg0[swir[0]], kind="enh", title="Reference B11 [1610 nm]",
               label="reflectance", px_m=C.PIXEL_M[SENSOR])
C.panel_label(axes[2], "c")

# (d) ch4
C.imshow_layer(axes[3], ch4, kind="enh", title="CH4 enhancement [ppb]",
               label="ppb", px_m=C.PIXEL_M[SENSOR])
C.overlay_mask(axes[3], plume, which="imeo")
C.panel_label(axes[3], "d")

# (e) plume mask
C.imshow_layer(axes[4], plume.astype(float), kind="prob", title="Plume mask",
               cbar=False, px_m=C.PIXEL_M[SENSOR])
C.overlay_mask(axes[4], plume, which="imeo")
C.panel_label(axes[4], "e")

# (f) DEM
C.imshow_layer(axes[5], dem, kind="dem", title="DEM [m]",
               label="m", px_m=C.PIXEL_M[SENSOR])
C.panel_label(axes[5], "f")

plt.show()

## The retrieval, computed here

The matched filter of MARS-S2L on the SWIR pair: the ratio between the target and a reference in the two methane-absorbing bands. `C.mbmp` is the same arithmetic the pipeline applies; the third panel uses the median of the four references, which is the multi-reference variant of the dataset.

In [ ]:
# MBMP for the anatomy scene.
swir = C.SENSORS[SENSOR]["swir"]          # 0-based indices (numpy)
t11, t12 = tgt[swir[0]], tgt[swir[1]]
b0 = C.read(s, "bg0")
d0 = -C.mbmp(t11, t12, b0[swir[0]], b0[swir[1]])

mbs = []
for k in range(4):
    bg = C.read(s, f"bg{k}")
    d = C.mbmp(t11, t12, bg[swir[0]], bg[swir[1]])
    mbs.append(d)
d_med = -np.nanmedian(mbs, axis=0)

mask = plume > 0

In [ ]:
fig, axes = C.panels(3, rows=1, kind="image")

C.imshow_layer(axes[0], d0, kind="mbmp", title="MBMP enhancement, sign flipped",
               label="dimensionless", sym=True, px_m=C.PIXEL_M[SENSOR])
C.overlay_mask(axes[0], mask, which="imeo")
C.panel_label(axes[0], "a")

C.imshow_layer(axes[1], ch4, kind="enh", title="CH4 enhancement from MARS [ppb]",
               label="ppb", px_m=C.PIXEL_M[SENSOR])
C.overlay_mask(axes[1], mask, which="imeo")
C.panel_label(axes[1], "b")

C.imshow_layer(axes[2], d_med, kind="mbmp", title="MBMP enhancement, median of 4 (sign flipped)",
               label="dimensionless", sym=True, px_m=C.PIXEL_M[SENSOR])
C.overlay_mask(axes[2], mask, which="imeo")
C.panel_label(axes[2], "c")

plt.show()

If you use these data, cite the dataset record and the TACO specification, both listed in the repository README.